# T-condition behavioural screen (Qwen2.5-VL-3B)

Plan: `docs/plan_v2.md` section 4, step 3 -- the cheapest kill-test. **Does an
incongruent note move the model's answer on >= ~50 objects (D41)?**

Thin wrapper: all logic lives in `src/circuit_overlap/`. **[FILL IN]** cells need
information the repo does not have; **[VERIFY]** cells check an assumption before
it can silently corrupt the run.

Dataset: `mgolov/Visual-Counterfact`, `split="color"`, 493 rows. Note its real
columns -- `object`, `correct_answer` (a *stringified list* of acceptable
colours), `incorrect_answer`, `original_image`, `counterfact_image`.

Settings: GPU T4, Internet ON.

## 1. Clone the repo and install

In [ ]:
# [FILL IN] your repo URL
REPO_URL = ""
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file and new names would appear
# to be missing.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

In [ ]:
# bitsandbytes is required for the 4-bit load (D10). transformers may cache
# "is bitsandbytes available?" at import time, so if the 4-bit load still
# fails right after installing, restart the kernel and run from cell 1.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| bf16 supported:", torch.cuda.is_bf16_supported())

# Visual-Counterfact is read with an HF token (Lietzow's README).
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 2. Build the object manifest

The T condition pastes notes on the **original_image** (real colour), and the
note then claims `incorrect_answer`.

`correct_answer` is a stringified list such as `"['brown', 'white']"` -- several
colours can be correct for one object. `parse_color_list` normalises it, and the
scorer treats "correct" as membership in that set. `object` is **not** assumed
unique, so ids get a row suffix while the prompt keeps the bare noun.

In [ ]:
import json, pathlib, inflect
from collections import Counter
from datasets import load_dataset
from circuit_overlap.data import parse_color_list, lietzow_object_name
from circuit_overlap.notes import object_bbox_from_pair

OUT = pathlib.Path("/kaggle/working/vc_color"); (OUT / "images").mkdir(parents=True, exist_ok=True)
_eng = inflect.engine()   # Lietzow pin inflect 7.5.0; lietzow_object_name
                          # applies their exact rule (capitalize, then require
                          # singular_noun(x) to differ from x).

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
print(ds.column_names, len(ds), "rows")

# Lietzow's loader drops invalid rows silently (contextlib.suppress(ValueError)
# around model_validate). Reproduce the same three rules, but count them.
items, reasons = [], Counter()
for i, row in enumerate(ds):
    try:
        name = lietzow_object_name(row["object"], inflect_engine=_eng)
    except ValueError as exc:
        reasons["plural" if "plural" in str(exc) else "extra info"] += 1; continue
    try:
        # require_list: their correct_answer is typed list[str], so a bare
        # string fails validation and the row disappears.
        correct = parse_color_list(row["correct_answer"], require_list=True)
        attack = parse_color_list(row["incorrect_answer"])[0]
    except (ValueError, TypeError):
        reasons["unparseable colours"] += 1; continue
    if attack in correct:                    # radish and spider, per their README
        reasons["colour overlap"] += 1; continue
    original = row["original_image"].convert("RGB")
    # The counterfactual image recolours the object and nothing else, so the
    # diff localises the object for free -- no segmentation model needed.
    bbox = object_bbox_from_pair(original, row["counterfact_image"].convert("RGB"))
    if bbox is None:
        reasons["no object box"] += 1          # kept, but placed by fallback anchor
    img_path = OUT / "images" / f"{name.replace(' ', '_')}_{i:04d}.png"
    original.save(img_path)
    items.append({"object_id": f"{name.replace(' ', '_')}_{i:04d}", "object_name": name,
                  "image": str(img_path), "original_colors": list(correct),
                  "counterfactual_color": attack, "object_bbox": bbox})

MANIFEST = OUT / "manifest.json"
MANIFEST.write_text(json.dumps(items, indent=1))
n_boxed = sum(e["object_bbox"] is not None for e in items)
print(f"kept {len(items)}  |  dropped {dict(reasons)}")
print(f"object box recovered for {n_boxed}/{len(items)} objects")
print("Lietzow report 469 valid objects, 467 after the colour-overlap rule.")

In [ ]:
from circuit_overlap.data import load_manifest, collect_color_vocabulary, check_colors_in_vocabulary

entries = load_manifest(MANIFEST)
vocab_from_data = collect_color_vocabulary(entries)
notes_cfg_raw = json.loads(open(f"{REPO_DIR}/configs/notes.json").read())
print(len(entries), "objects")
print("colours in the data:", vocab_from_data)
print("multi-colour objects:", sum(len(e.original_colors) > 1 for e in entries))
print("not in configs/notes.json placeholder vocabulary:",
      check_colors_in_vocabulary(entries, notes_cfg_raw["color_vocabulary"]))
# run_screen unions these automatically, but paste vocab_from_data into
# configs/notes.json so the committed config matches the data.

## 2b. [VERIFY] The 493-vs-467 question

Lietzow et al. report **469** objects (467 after dropping radish and spider);
this split has **493 rows**. Until that is explained, a head-set mismatch at the
replication gate cannot be attributed to either the dataset or the pipeline
(STATE.md, D20). Three candidate explanations, cheapest first.

In [ ]:
# The cascade above is the answer to the 493-vs-467 question, if the numbers land:
#   493 rows -> minus plural / parenthetical names -> 469 -> minus colour overlap -> 467.
# Lietzow's loader applies exactly these rules (src/vdpo/types/models.py) and
# swallows the failures, so the paper's counts are post-filter.
objs = [str(o).strip().lower() for o in ds["object"]]
dupes = {o: c for o, c in Counter(objs).items() if c > 1}
print(f"rows {len(objs)} | unique object names {len(set(objs))} | repeated {len(dupes)}")
print("  repeated examples:", sorted(dupes.items(), key=lambda kv: -kv[1])[:5])
print("  drop reasons:", dict(reasons), "-> kept", len(items))

# How many objects have more than one acceptable colour? (feeds D29a)
print("objects by number of acceptable colours:",
      dict(sorted(Counter(len(e["original_colors"]) for e in items).items())))

# If the counts still do not reach 469/467, check whether an earlier Hub
# revision differs:
from huggingface_hub import list_repo_commits
for c in list_repo_commits("mgolov/Visual-Counterfact", repo_type="dataset")[:10]:
    print(f"  {str(c.created_at)[:10]}  {c.commit_id[:8]}  {c.title[:60]}")

### 2c. [DEBUG] Why 463 and not 467?

Observed on 2026-10-19: our filters keep **463** (dropping 20 plural, 8 extra
info, 2 colour overlap) where Lietzow report **469** valid and **467** after the
colour-overlap rule. The colour-overlap count matches exactly, so the gap is 4
object names we reject and they keep.

Leading hypothesis: their `_sanitize_object_name` **capitalizes** the name
before the plural test (`singular_noun("Scissors")`), while we test lowercase.
Second hypothesis: `inflect` version — their lockfile pins **7.5.0**.

This cell costs no GPU and decodes no images (`ds["object"]` reads one column).

In [ ]:
from importlib.metadata import version
print("inflect", version("inflect"), "(Lietzow pin: 7.5.0)")
drops = {k: v for k, v in reasons.items() if k != "no object box"}
print(f"kept {len(items)}  |  dropped {drops}  |  Lietzow: 469 valid, 467 used")

# Settled 2026-10-19/20:
#  * capitalising before the plural test gives 469 (was 463)
#  * correct_answer is a list in every row  -> that hypothesis is dead
#  * no row has a path-only image           -> that hypothesis is dead
#  * the 2 colour-overlap drops are exactly radish and spider, as their README says
# Remaining: we keep 469 where they use 467.
#
# Last and most likely explanation: the dataset GREW. Their numbers would then
# describe a pre-July-2025 revision. Split sizes come from metadata, so this
# costs no download.
from datasets import load_dataset_builder
from huggingface_hub import list_repo_commits

for c in list_repo_commits("mgolov/Visual-Counterfact", repo_type="dataset"):
    try:
        info = load_dataset_builder("mgolov/Visual-Counterfact",
                                    revision=c.commit_id).info
        sizes = {k: v.num_examples for k, v in (info.splits or {}).items()}
    except Exception as exc:                      # revision may lack metadata
        sizes = f"unavailable ({type(exc).__name__})"
    print(f"  {str(c.created_at)[:10]}  {c.commit_id[:8]}  {str(sizes):50s} {c.title[:40]}")

print("\nIf a pre-July revision has color=469, the cascade is fully explained: "
      "their 469/467 is that release, ours is the current one. Then DECIDE (D20): "
      "pin that revision for the replication gate, or use current and disclose "
      "the difference. If nothing matches, stop chasing 2 objects (0.4%) and "
      "record the deviation in RESULTS.md.")

## 3. [VERIFY] Look at one attack image before spending GPU time

In [ ]:
from circuit_overlap.notes import render_note, variant_texts, matched_neutral_word
from circuit_overlap.tokens import mask_to_token_indices, token_grid_shape
from circuit_overlap.t_screen import load_notes_config

cfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")
e = entries[0]
img = e.open_image()
print(e.object_name, "| correct:", e.original_colors, "| note says:", e.counterfactual_color)
print("object box:", e.object_bbox)
texts = variant_texts(e.primary_original_color, e.counterfactual_color,
                      matched_neutral_word(e.counterfactual_color, cfg["neutral_words"]))
for spec in cfg["specs"]:
    r = render_note(img, texts["incongruent"], spec, bbox=e.object_bbox)
    occ = r.object_occlusion(e.object_bbox)
    print(f"{spec.name:17s} mode={spec.mode:7s} area={r.area_frac:.3f} "
          f"occlusion={occ:.0%} font={r.font_name} {r.font_size_px}px")
    display(r.image.resize((320, 320)))

# Occlusion is the number to watch: a note covering the object removes the
# colour evidence, so the model following it is absence of evidence, not
# typographic override. Scan a few objects before committing GPU time:
occs = []
for e2 in entries[:30]:
    rr = render_note(e2.open_image(), "blue", cfg["specs"][1], bbox=e2.object_bbox)
    occs.append(rr.object_occlusion(e2.object_bbox))
import numpy as np
print(f"\non_object_medium occlusion over 30 objects: "
      f"median {np.nanmedian(occs):.0%}, max {np.nanmax(occs):.0%}")

In [ ]:
# [VERIFY] mask -> image-token mapping against the processor's real grid.
# The processor RESIZES the image, so the mask must be mapped onto the grid the
# model actually sees, not onto the source resolution.
from transformers import AutoProcessor
from circuit_overlap.t_screen import ScreenConfig
from circuit_overlap.tokens import merged_grid_from_grid_thw, mask_to_token_indices_on_grid

config = ScreenConfig()        # [VERIFY] checkpoint id
processor = AutoProcessor.from_pretrained(config.checkpoint, use_fast=True,
                                          min_pixels=config.pixels, max_pixels=config.pixels)
enc = processor(text=["x"], images=[r.image], return_tensors="pt")
grid_thw = enc["image_grid_thw"]
rows, cols = merged_grid_from_grid_thw(grid_thw)
print("source image :", r.image.size, "(w, h)")
print("patch grid   :", grid_thw.tolist(), "-> merged token grid", (rows, cols),
      "=", rows * cols, "image tokens")

idx, grid = mask_to_token_indices_on_grid(r.mask, rows, cols, return_grid=True)
print(f"note covers {idx.size}/{rows*cols} image tokens "
      f"({idx.size/(rows*cols):.1%}); mask area {r.area_frac:.1%} of the frame")
# Sanity: the token share should track the pixel share, since the resize is
# uniform. A large gap means the mask and the grid are misaligned.
assert abs(idx.size/(rows*cols) - r.area_frac) < 0.05, "mask/grid mismatch"
print("mask -> token mapping agrees with the pixel area")

## 4. [VERIFY] Colour words must be single, distinct first tokens (D26)

In [ ]:
from circuit_overlap.t_screen import color_first_token_ids
ids, collisions = color_first_token_ids(processor.tokenizer, vocab_from_data)
print("first-token ids (both case variants, no leading space, as Lietzow do):")
for c, t in ids.items():
    print(f"  {c:8s} {t}")
print("colours sharing a token id:", collisions)
# A collision makes the logit difference meaningless: drop those colours, or
# the objects using them, and say so in RESULTS.md.

## 5. Smoke run on 5 objects, then the full screen

In [ ]:
from circuit_overlap.t_screen import load_model, run_screen
model, processor = load_model(config)
records, summary = run_screen(entries[:5], model, processor, config, cfg)

# Delta = logit(best accepted colour) - logit(note's colour). The attack flips
# the answer only when Delta goes negative, so the number that matters is not
# the shift but what is LEFT: margin = Delta after the attack. A small positive
# margin means more salience could flip it; a large one means it cannot.
print(f"{'layout':17s} {'object':18s} {'d_neutral':>9s} {'d_attack':>9s} "
      f"{'shift':>7s} {'margin':>7s}  note-colour rank  occlusion")
for r in records:
    cl = r.color_logits_incongruent or {}
    rank = 1 + sorted(cl.values(), reverse=True).index(cl[r.counterfactual_color]) if cl else -1
    print(f"{r.layout:17s} {r.object_name[:18]:18s} {r.delta_neutral:9.2f} "
          f"{r.delta_incongruent:9.2f} {r.delta_neutral - r.delta_incongruent:+7.2f} "
          f"{r.delta_incongruent:+7.2f}  {rank:>16d}  {r.object_occlusion:.0%}")

import numpy as np
margins = [r.delta_incongruent for r in records]
print(f"\nmargin to a flip: median {np.median(margins):+.2f}, min {np.min(margins):+.2f}")
print("Rule of thumb: a median margin under ~2 logits means salience knobs are "
      "worth trying; over ~8 means the argmax criterion is the wrong one for "
      "this model and the continuous variant (plan section 4) is the fallback.")

In [ ]:
# If the margin is small, try salience knobs on the same 5 objects before
# committing to a full run. Each is a one-line change in configs/notes.json.
from dataclasses import replace
from circuit_overlap.notes import NoteSpec

variants = {
    "baseline":      cfg["specs"][1],
    "uppercase":     replace(cfg["specs"][1], uppercase=True, name="uppercase"),
    "bigger":        replace(cfg["specs"][1], rel_width=0.36, name="bigger"),
    "high_contrast": replace(cfg["specs"][1], bg_color=(255, 255, 255),
                             text_color=(0, 0, 0), name="high_contrast"),
}
for label, spec in variants.items():
    recs, _ = run_screen(entries[:5], model, processor, config,
                         {**cfg, "specs": [spec]})
    margins = [x.delta_incongruent for x in recs]
    fooled = sum(1 for x in recs if x.delta_incongruent < 0)
    print(f"{label:14s} median margin {np.median(margins):+6.2f}  "
          f"flipped {fooled}/{len(recs)}  occlusion {recs[0].object_occlusion:.0%}")

In [ ]:
# Full run. Checkpoints one JSON line per object to /kaggle/working, so an
# interrupted session loses at most one object (D44).
#
# RUN THIS AS A BACKGROUND JOB: Save Version -> "Save & Run All (Commit)".
# It runs headless for up to 12 h and keeps /kaggle/working as the version
# output, so closing the laptop cannot kill it.
#
# To RESUME a previous attempt: attach that version's output as an input
# dataset and point CKPT at it (a fresh session starts with an empty
# /kaggle/working, so the checkpoint must be carried in deliberately).
import time
CKPT = "/kaggle/working/t_screen_checkpoint.jsonl"

t0 = time.time()
records, summary = run_screen(entries, model, processor, config, cfg,
                              checkpoint_path=CKPT)
print(f"\n{time.time()-t0:.0f}s for {len(records)} records "
      f"({len(cfg['specs'])} layouts x {len(config.variants)} variants)")
{k: v for k, v in summary.items() if k not in ("usable_object_ids", "color_vocabulary")}

## 6. Save the result JSON (download it and commit it to `results/`)

In [ ]:
from dataclasses import asdict
from circuit_overlap.provenance import result_envelope, write_result

envelope = result_envelope(
    experiment="t_behavioural_screen",
    payload={"summary": summary, "records": [asdict(r) for r in records]},
    model="Qwen2.5-VL-3B",
    checkpoint=config.checkpoint,
    seed=config.seed,
    prompt=config.prompt_template,
    n=summary["n_objects"],
    repo_dir=REPO_DIR,
    dataset={"source_id": "mgolov/Visual-Counterfact", "split": "color",
             "n_rows_total": len(ds), "n_after_exclusions": len(entries),
             "excluded": sorted(EXCLUDE), "drop_reasons": dict(reasons)},
    layouts=[s.name for s in cfg["specs"]],
    variants=list(config.variants),
    dtype=config.dtype, load_in_4bit=config.load_in_4bit, pixels=config.pixels,
    bf16_available=bool(torch.cuda.is_bf16_supported()),
)
write_result("/kaggle/working/t_screen_qwen25vl3b.json", envelope)

# The usable-object list is the example set for T-circuit discovery, so write
# it on its own: it is small, belongs in git, and losing it means re-running
# the whole screen.
write_result("/kaggle/working/t_usable_objects.json", result_envelope(
    experiment="t_usable_objects",
    payload={"usable_object_ids": summary["usable_object_ids"],
             "n_usable": summary["n_objects_usable"],
             "n_objects": summary["n_objects"],
             "fool_rate_among_eligible": summary["fool_rate_among_eligible"]},
    model="Qwen2.5-VL-3B", checkpoint=config.checkpoint, seed=config.seed,
    prompt=config.prompt_template, n=summary["n_objects_usable"], repo_dir=REPO_DIR,
))

print("usable objects:", summary["n_objects_usable"],
      "| gate_passed:", summary["gate_passed"])
print("continuous criterion:", summary["continuous_fallback"])
print("\nDownload both JSONs, commit them to results/, and add a row to "
      "docs/RESULTS.md. t_usable_objects.json is the input to circuit discovery.")

## 7. What the outcome means (plan section 4)

| Result | Next |
|---|---|
| `gate_passed = True` (>= 50 usable objects) | Proceed to T-circuit discovery |
| Few flips but `continuous_fallback.n_objects_shifted` healthy | Use the continuous variant: patch on logit-difference shifts without an argmax flip |
| Few flips and little shift | Raise note salience (size, contrast) in `configs/notes.json`, or try PaliGemma-3B as primary |

Then record the number in `docs/RESULTS.md` with model, n, seed, prompt and
commit, and log the 493-row / Lietzow-467 discrepancy in `docs/STATE.md`.